# Deep Learning with PyTorch

We will build a small model that predicts whether a support ticket is urgent. The same example is used from the first tensor to the final prediction.

## What you will do

1. Represent features and labels.
2. Calculate one neuron by hand.
3. Convert a raw score into a probability.
4. Understand ReLU and a small network.
5. Compare simple loss functions.
6. See weighted and custom loss.
7. Train and evaluate a tiny neural network.
8. Convert the model score into a safe system action.

Run the cells from top to bottom.

## 1. Setup

Import PyTorch, fix the random seed, and check whether the code will use CPU, CUDA, or Apple MPS.

In [1]:
# Use a fixed seed so the results stay the same on each run.

import torch
from torch import nn

torch.manual_seed(7)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Using device:", device)

PyTorch version: 2.2.0
Using device: mps


## 2. Features and labels

Store two support tickets and their correct labels as tensors.

The two features are:

- negative-word score from 0 to 1,
- waiting-time score from 0 to 1.

The label is `1` for urgent and `0` for normal.

In [2]:
# Each row is one ticket. Each column is one feature.

import torch

features = torch.tensor([
    [0.90, 0.80],  # ticket 1: negative and waiting a long time
    [0.10, 0.20],  # ticket 2: calm and recently created
], dtype=torch.float32)

labels = torch.tensor([
    [1.0],  # urgent
    [0.0],  # normal
], dtype=torch.float32)

print("Features:\n", features)
print("Labels:\n", labels)
print("Feature shape:", features.shape)
print("Label shape:", labels.shape)
print("Feature dtype:", features.dtype)

Features:
 tensor([[0.9000, 0.8000],
        [0.1000, 0.2000]])
Labels:
 tensor([[1.],
        [0.]])
Feature shape: torch.Size([2, 2])
Label shape: torch.Size([2, 1])
Feature dtype: torch.float32


## 3. Calculate one neuron

A neuron multiplies each input by a weight, adds the results, and then adds a bias.

```text
negative score × weight 1 ─┐
                           ├─ add + bias ─> raw score
waiting score  × weight 2 ─┘
```

In [3]:
# Calculate the raw score of one neuron.

negative_score = 0.8
waiting_score = 0.6

weight_negative = 2.0
weight_waiting = 1.0
bias = -1.0

negative_contribution = negative_score * weight_negative
waiting_contribution = waiting_score * weight_waiting
raw_score = negative_contribution + waiting_contribution + bias

print("Negative-word contribution:", negative_contribution)
print("Waiting-time contribution:", waiting_contribution)
print("Bias:", bias)
print("Raw score / logit:", raw_score)

Negative-word contribution: 1.6
Waiting-time contribution: 0.6
Bias: -1.0
Raw score / logit: 1.2000000000000002


## 4. Convert a logit into a probability

A logit can be any number. Sigmoid changes it to a value between `0` and `1`.

```text
raw score 1.2 ─> sigmoid ─> probability about 0.77
```

In [4]:
# Compare logits with their sigmoid probabilities.

import torch

logits = torch.tensor([-3.0, 0.0, 1.2, 3.0])
probabilities = torch.sigmoid(logits)

for logit, probability in zip(logits, probabilities):
    print(f"logit {logit.item():>4.1f} -> probability {probability.item():.3f}")

logit -3.0 -> probability 0.047
logit  0.0 -> probability 0.500
logit  1.2 -> probability 0.769
logit  3.0 -> probability 0.953


## 5. Hidden layer and ReLU

ReLU changes negative values to zero and keeps positive values. This helps hidden layers learn useful patterns.

```text
2 inputs -> Linear layer -> ReLU -> Linear layer -> 1 logit
```

In [5]:
# First check ReLU, then build a network with one hidden layer.

import torch
from torch import nn

values = torch.tensor([-3.0, -1.0, 0.0, 2.0, 5.0])
relu_values = torch.relu(values)

print("Before ReLU:", values.tolist())
print("After ReLU: ", relu_values.tolist())

model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
)

example_batch = torch.tensor([[0.9, 0.8], [0.1, 0.2]])
logits = model(example_batch)

print("\nModel:\n", model)
print("Input shape:", example_batch.shape)
print("Output shape:", logits.shape)

Before ReLU: [-3.0, -1.0, 0.0, 2.0, 5.0]
After ReLU:  [0.0, 0.0, 0.0, 2.0, 5.0]

Model:
 Sequential(
  (0): Linear(in_features=2, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)
Input shape: torch.Size([2, 2])
Output shape: torch.Size([2, 1])


## 6. Loss for regression: MAE and MSE

MAE and MSE are common losses for predicting numbers. MSE gives more importance to large mistakes.

In [6]:
# Compare absolute error with squared error.

import torch

actual_hours = torch.tensor([5.0, 5.0])
predicted_hours = torch.tensor([7.0, 15.0])

absolute_errors = torch.abs(predicted_hours - actual_hours)
mae = absolute_errors.mean()
mse = ((predicted_hours - actual_hours) ** 2).mean()

print("Absolute errors:", absolute_errors.tolist())
print("MAE:", mae.item())
print("MSE:", mse.item())
print("MSE is much larger because the 10-hour error is squared.")

Absolute errors: [2.0, 10.0]
MAE: 6.0
MSE: 52.0
MSE is much larger because the 10-hour error is squared.


## 7. Binary Cross-Entropy (BCE) for urgent / normal classification

### What BCE does

A binary classifier has **two possible labels**:

| Ticket type | Training label (`y`) |
|---|---:|
| urgent | `1` |
| normal | `0` |

The model first returns a **logit** (`z`). Sigmoid changes this raw score into a probability:

```text
p = sigmoid(z) = 1 / (1 + e^(-z))
```

- a large positive logit means the model leans towards **urgent**;
- a large negative logit means the model leans towards **normal**;
- a logit of `0` becomes probability `0.5`, so the model is unsure.

BCE gives a small loss for a good prediction and a large loss for a bad prediction:

```text
BCE = -[y × log(p) + (1 - y) × log(1 - p)]
```

Remember these points:

- For an urgent ticket, a low urgent probability gives a high loss.
- For a normal ticket, a high urgent probability gives a high loss.
- A strong correct prediction gets loss close to `0`. A strong wrong prediction gets a large loss.

### Why use `BCEWithLogitsLoss`?

PyTorch's `nn.BCEWithLogitsLoss` safely combines sigmoid and BCE. Give it raw logits, not sigmoid probabilities.

```python
# Correct: raw model output
loss = nn.BCEWithLogitsLoss()(logits, labels)

# Do not apply sigmoid before this loss:
# loss = nn.BCEWithLogitsLoss()(torch.sigmoid(logits), labels)  # incorrect
```

Use `torch.sigmoid(logits)` later when you want to display a probability or apply a threshold.

In [7]:
# All three tickets are urgent. Only the model score changes.

import torch
from torch import nn

# reduction="none" gives one loss value for each ticket.
loss_function = nn.BCEWithLogitsLoss(reduction="none")

logits = torch.tensor([[2.2], [0.0], [-2.2]])
labels = torch.tensor([[1.0], [1.0], [1.0]])

probabilities = torch.sigmoid(logits)  # convert scores for display
losses = loss_function(logits, labels)  # the loss receives raw logits

for logit, probability, label, loss in zip(logits, probabilities, labels, losses):
    print(
        f"logit={logit.item():>4.1f}  "
        f"urgent_probability={probability.item():.3f}  "
        f"label={label.item():.0f}  loss={loss.item():.3f}"
    )

print("\nAt 50% probability, loss is 0.693. A lower urgent probability gives a larger loss.")

logit= 2.2  urgent_probability=0.900  label=1  loss=0.105
logit= 0.0  urgent_probability=0.500  label=1  loss=0.693
logit=-2.2  urgent_probability=0.100  label=1  loss=2.305

At 50% probability, loss is 0.693. A lower urgent probability gives a larger loss.


## 8. Positive-weighted BCE for rare urgent tickets

### What does `pos_weight=4` mean?

`4.0` is a weight. It tells the loss:

> When an urgent ticket is predicted badly, make its loss four times larger.

For one binary output, PyTorch expects one weight value, so we write `torch.tensor([4.0])`.

The calculation is:

```text
weighted BCE = -[positive weight × y × log(p) + (1 - y) × log(1 - p)]
```

The weight changes only examples whose true label is `1`. It does not:

- change an urgent label from `1` to `4`;
- multiply every example's loss by four;
- directly change the model's probability;
- replace the prediction threshold.

In the next example, normal BCE is about `1.313`. With weight `4`, it becomes about `5.253`. The model gets a stronger signal to correct the mistake.

### Why might we choose 4?

A common starting point is:

```text
pos_weight = number of normal examples / number of urgent examples
```

For example, 80 normal tickets and 20 urgent tickets give `80 / 20 = 4`.

This is only a starting point. Check precision and recall on validation data. A very high weight may catch more urgent tickets but also create more false alerts.

In [8]:
# Compare normal BCE with BCE that gives urgent labels four times more weight.

import torch
from torch import nn

one_logit = torch.tensor([[-1.0]])  # sigmoid(-1) ≈ 0.269: model currently leans "normal"
urgent_label = torch.tensor([[1.0]])

normal_bce = nn.BCEWithLogitsLoss()
weighted_bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([4.0]))

normal_loss = normal_bce(one_logit, urgent_label)
weighted_loss = weighted_bce(one_logit, urgent_label)

print("Predicted urgent probability:", round(torch.sigmoid(one_logit).item(), 3))
print("Normal BCE loss:         ", round(normal_loss.item(), 3))
print("Weighted BCE loss:       ", round(weighted_loss.item(), 3))
print("Weight check:            ", round(weighted_loss.item() / normal_loss.item(), 1), "x")
print("\nHere, 4 means: this true urgent example contributes 4x its normal BCE loss.")

Predicted urgent probability: 0.269
Normal BCE loss:          1.313
Weighted BCE loss:        5.253
Weight check:             4.0 x

Here, 4 means: this true urgent example contributes 4x its normal BCE loss.


### Does the weight affect a normal ticket?

No. `pos_weight` looks at the true label, not the sign of the logit.

The next example uses the same logit (`-1.0`) twice:

- For an urgent ticket (`label = 1`), this prediction is a mistake, so its loss is multiplied by `4`.
- For a normal ticket (`label = 0`), the loss stays unchanged.

Here, positive means the class with `label = 1`. It does not mean a positive logit.

In [9]:
# The weight changes urgent-label loss but leaves normal-label loss unchanged.

same_logit = torch.tensor([[-1.0], [-1.0]])
true_labels = torch.tensor([[1.0], [0.0]])  # first urgent, second normal
names = ["Urgent ticket (missed)", "Normal ticket (correct)"]

unweighted_losses = nn.BCEWithLogitsLoss(reduction="none")(same_logit, true_labels)
weighted_losses = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([4.0]), reduction="none"
)(same_logit, true_labels)

for name, label, normal, weighted in zip(names, true_labels, unweighted_losses, weighted_losses):
    print(f"{name}: label={label.item():.0f}  normal={normal.item():.3f}  weighted={weighted.item():.3f}")

Urgent ticket (missed): label=1  normal=1.313  weighted=5.253
Normal ticket (correct): label=0  normal=0.313  weighted=0.313


### When to use weighted BCE — and what to check

Use positive weighting when positive examples are rare or expensive to miss. Examples include fraud, safety defects, and urgent tickets.

After training, check:

1. **Recall:** Of truly urgent tickets, how many did we catch?
2. **Precision:** Of tickets predicted urgent, how many were really urgent?
3. **False negatives:** Which urgent tickets did we still miss?
4. **False positives:** How many normal tickets need extra review?

`pos_weight` changes training. The threshold changes the final decision. Check them separately.

## 9. A simple custom loss combination

The model can learn urgency as the main task and category as a smaller helper task.

```text
total loss = urgency loss + 0.2 × category loss
```

Start with standard losses. Keep a custom combination only when validation results improve.

In [10]:
# Combine the main urgency loss with a smaller category loss.

import torch
from torch import nn

urgency_logit = torch.tensor([[0.4]], requires_grad=True)
urgency_label = torch.tensor([[1.0]])

category_logits = torch.tensor([[0.2, 1.1, -0.4]], requires_grad=True)
category_label = torch.tensor([1])  # class 1 is correct

urgency_loss = nn.BCEWithLogitsLoss()(urgency_logit, urgency_label)
category_loss = nn.CrossEntropyLoss()(category_logits, category_label)

total_loss = urgency_loss + 0.2 * category_loss
total_loss.backward()

print("Urgency loss:", round(urgency_loss.item(), 3))
print("Category loss:", round(category_loss.item(), 3))
print("Total loss:   ", round(total_loss.item(), 3))
print("Urgency gradient exists:", urgency_logit.grad is not None)
print("Category gradient exists:", category_logits.grad is not None)

Urgency loss: 0.513
Category loss: 0.488
Total loss:    0.611
Urgency gradient exists: True
Category gradient exists: True


## 10. Train a tiny neural network

Create support-ticket data and run the complete training cycle: predict, calculate loss, run backward, and update the model.

The labels follow a rule based on negative words and waiting time. The model does not see the rule; it learns from examples.

In [11]:
# Build data, split it, and train a small model.

import torch
from torch import nn

torch.manual_seed(7)

# 1. Create 500 support-ticket examples with two features.
features = torch.rand(500, 2)
hidden_urgency_score = 1.4 * features[:, 0] + features[:, 1]
labels = (hidden_urgency_score > 1.25).float().unsqueeze(1)

# 2. Keep separate training and validation examples.
train_x, val_x = features[:400], features[400:]
train_y, val_y = labels[:400], labels[400:]

# 3. Create a small neural network.
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)

# 4. Choose the loss and optimiser.
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

# 5. Repeat the learning steps.
for epoch in range(101):
    model.train()
    optimizer.zero_grad()
    train_logits = model(train_x)
    train_loss = loss_function(train_logits, train_y)
    train_loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        model.eval()
        with torch.inference_mode():
            val_logits = model(val_x)
            val_loss = loss_function(val_logits, val_y)
        print(
            f"epoch={epoch:3d}  "
            f"train_loss={train_loss.item():.4f}  "
            f"val_loss={val_loss.item():.4f}"
        )

epoch=  0  train_loss=0.7633  val_loss=0.6919
epoch= 20  train_loss=0.6642  val_loss=0.6897
epoch= 40  train_loss=0.5597  val_loss=0.5790
epoch= 60  train_loss=0.3499  val_loss=0.3271
epoch= 80  train_loss=0.2052  val_loss=0.1835
epoch=100  train_loss=0.1428  val_loss=0.1242


## 11. Probability, threshold, and metrics

Turn validation logits into probabilities and compare two thresholds. A lower threshold may catch more urgent tickets, but it may also create more false alerts.

Run the training cell first because this cell evaluates that trained model.

In [12]:
# Compare accuracy, precision, and recall at two thresholds.

import torch

def binary_metrics(labels, predictions):
    labels = labels.bool()
    predictions = predictions.bool()

    true_positive = (predictions & labels).sum().item()
    false_positive = (predictions & ~labels).sum().item()
    false_negative = (~predictions & labels).sum().item()

    accuracy = (predictions == labels).float().mean().item()
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    return accuracy, precision, recall

model.eval()
with torch.inference_mode():
    probabilities = torch.sigmoid(model(val_x))

for threshold in [0.50, 0.70]:
    predictions = probabilities >= threshold
    accuracy, precision, recall = binary_metrics(val_y, predictions)
    print(
        f"threshold={threshold:.2f}  accuracy={accuracy:.3f}  "
        f"precision={precision:.3f}  recall={recall:.3f}"
    )

threshold=0.50  accuracy=0.970  precision=0.982  recall=0.964
threshold=0.70  accuracy=0.950  precision=1.000  recall=0.911


## 12. Use the model inside a safe system

Use the probability only to choose a support queue. Important actions such as refunds or account changes still need normal system checks.

In [13]:
# Use the urgency probability to choose a support queue.

import torch

new_ticket = torch.tensor([[0.85, 0.75]])

model.eval()
with torch.inference_mode():
    logit = model(new_ticket)
    urgency_probability = torch.sigmoid(logit).item()

if urgency_probability >= 0.70:
    action = "Route to an urgent human-review queue"
else:
    action = "Keep in the normal support queue"

print("Urgency probability:", round(urgency_probability, 3))
print("Allowed action:", action)
print("Not allowed from this score alone: issue a refund or change an account")

Urgency probability: 0.986
Allowed action: Route to an urgent human-review queue
Not allowed from this score alone: issue a refund or change an account


## 13. Practice

Try these changes one at a time:

1. Change the hidden layer from 8 neurons to 4. Does validation loss change?
2. Change the learning rate from `0.03` to `0.003`. Does learning become slower?
3. Try thresholds `0.40`, `0.60`, and `0.80`. Compare precision and recall.
4. In the custom loss cell, change the category weight from `0.2` to `1.0`. Which task now has more influence?

### Key points

- Features are the inputs; labels are the correct answers.
- Weights and bias create a logit. Sigmoid changes it into a probability.
- Loss measures the training error.
- The training order is: predict → loss → backward → update.
- Loss and threshold have different jobs.
- A model score should not directly perform an important account action.